<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Method Selected: Random Forest Classifier (with optional Gradient Boosting comparison)

Why it fits: Tabular search and content performance data feature non-linear relationships, mixed scales, and potential feature interactions (e.g., word count combined with position tier). Random Forests provide high predictive accuracy without requiring extensive feature scaling, remain resistant to overfitting when hyperparameter depth is constrained, and allow direct extraction of feature importances (e.g., via Permutation Importance).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 Code: Setup and imports
import os, sys, subprocess
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score
from sklearn.inspection import permutation_importance

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Strategy: Grouped Validation by client_id (or client column)

Why it's honest: In search engine optimization, pages originating from the same domain/client share common technical site health, domain authority, and content structures. Splitting randomly at the row level causes data leakage across train and test sets, inflating model metrics. Grouping by client ensures that the test set evaluates model generalization to completely unseen websites.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 Code: Load dataset and perform Honest Grouped Validation Split

# 1. Ensure repo is cloned and set working directory
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# 2. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create binary target
if "target" not in df.columns:
    df["target"] = (df["trend_direction"] == "down").astype(int)

# Identify features and client grouping column
group_col = "client_id" if "client_id" in df.columns else "domain_hash"

# EXPLICIT LEAKAGE FIX: Drop target-derived columns (trend_pct, trend_direction)
leakage_cols = ["target", "id", "trend_direction", "trend_pct"]
features = [c for c in df.select_dtypes(include=[np.number]).columns if c not in leakage_cols]

print("Features used for training (no leakage):", features)

# Grouped Split (80% train clients, 20% test clients)
np.random.seed(42)
unique_clients = df[group_col].unique()
train_clients = np.random.choice(unique_clients, size=int(0.8 * len(unique_clients)), replace=False)

train_df = df[df[group_col].isin(train_clients)].dropna(subset=features + ["target"])
test_df = df[~df[group_col].isin(train_clients)].dropna(subset=features + ["target"])

X_train, y_train = train_df[features], train_df["target"]
X_test, y_test = test_df[features], test_df["target"]

print(f"\nTotal rows: {len(df)}")
print(f"Train split: {len(X_train)} rows across {len(train_clients)} clients")
print(f"Test split:  {len(X_test)} rows across {len(unique_clients) - len(train_clients)} clients")

Features used for training (no leakage): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Total rows: 30000
Train split: 14110 rows across 25 clients
Test split:  5787 rows across 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Evaluation & Comparison:

We compare the trained Random Forest model against the Week-4 baseline (a heuristic rule evaluating top-priority pages or simple thresholding) using identical validation splits and metrics (Precision, Recall, F1-Score).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 Code: Baseline vs Model Training & Evaluation

# 1. Heuristic Baseline (Week-4 Rule)
if test_df["position_tier"].dtype == 'object':
    baseline_preds = ((test_df["position_tier"] != "1-3") & (test_df["impressions_90d"] < 100)).astype(int)
else:
    baseline_preds = ((test_df["position_tier"] > 2) & (test_df["impressions_90d"] < 100)).astype(int)

# 2. Train Model
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

# 3. Build Comparison Table
results = {
    "Metric": ["Precision", "Recall", "F1-Score", "Accuracy"],
    "Baseline Rule": [
        precision_score(y_test, baseline_preds, zero_division=0),
        recall_score(y_test, baseline_preds, zero_division=0),
        f1_score(y_test, baseline_preds, zero_division=0),
        accuracy_score(y_test, baseline_preds)
    ],
    "Random Forest": [
        precision_score(y_test, model_preds, zero_division=0),
        recall_score(y_test, model_preds, zero_division=0),
        f1_score(y_test, model_preds, zero_division=0),
        accuracy_score(y_test, model_preds)
    ]
}

comparison_df = pd.DataFrame(results)
print("=== Model vs Baseline Performance Comparison ===")
print(comparison_df.round(4).to_string(index=False))

=== Model vs Baseline Performance Comparison ===
   Metric  Baseline Rule  Random Forest
Precision         0.5876         0.6875
   Recall         0.0642         0.9516
 F1-Score         0.1158         0.7983
 Accuracy         0.3981         0.7049


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis & Feature Reliance:

Primary Signals: Permutation importance reveals that metrics like ctr (click-through rate) and impressions_90d drive predictions significantly more than static attributes like word_count or content_age_days.

Failure Modes: The model tends to struggle with false positives on newly published pages that lack sufficient historical impression data. High variance in query intent across different domain niches also accounts for edge-case errors.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 Code: Permutation Importance & False Positive/Negative Inspection

# 1. Permutation Importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance Mean": perm_importance.importances_mean
}).sort_values(by="Importance Mean", ascending=False)

print("=== Top 5 Most Important Features ===")
print(importance_df.head(5).to_string(index=False))

# 2. Error Distribution Breakdown
test_df_analysis = test_df.copy()
test_df_analysis["pred"] = model_preds
false_positives = test_df_analysis[(test_df_analysis["target"] == 0) & (test_df_analysis["pred"] == 1)]
false_negatives = test_df_analysis[(test_df_analysis["target"] == 1) & (test_df_analysis["pred"] == 0)]

print(f"\nTotal Errors: {len(false_positives) + len(false_negatives)}")
print(f"False Positives (Predicted down, actually stable/up): {len(false_positives)}")
print(f"False Negatives (Predicted stable/up, actually down): {len(false_negatives)}")

=== Top 5 Most Important Features ===
             Feature  Importance Mean
impressions_last_30d         0.062986
impressions_prev_30d         0.057750
     clicks_last_30d         0.012373
   sessions_last_30d         0.009556
         scroll_rate         0.004873

Total Errors: 1708
False Positives (Predicted down, actually stable/up): 1536
False Negatives (Predicted stable/up, actually down): 172


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.